###  1️⃣ Advanced Joins & Set Operations

**Concept**

Beyond INNER/LEFT joins, advanced joins let you compare whole datasets, join a table to itself, or return all possible combinations.

In [0]:
-- Self join: employees who share the same manager
SELECT e1.FirstName AS Employee, e2.FirstName AS Colleague
FROM Employees e1
JOIN Employees e2
  ON e1.ManagerID = e2.ManagerID
 AND e1.EmployeeID <> e2.EmployeeID;

-- FULL OUTER JOIN: everything from both tables
SELECT c.CustomerName, o.OrderID
FROM Customers c
FULL OUTER JOIN Orders o
  ON c.CustomerID = o.CustomerID;

-- INTERSECT & EXCEPT (SQL Server/Postgres)
SELECT City FROM Customers
INTERSECT
SELECT City FROM Suppliers;  -- only cities in both


###  2️⃣ Window (Analytic) Functions

**Concept**

Window functions calculate values across sets of rows without collapsing them like GROUP BY does.

In [0]:
-- Ranking employees by salary
SELECT EmployeeID, Salary,
  RANK() OVER (ORDER BY Salary DESC) AS SalaryRank
FROM Employees;

-- Running total per customer
SELECT CustomerID, OrderDate, Amount,
  SUM(Amount) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS RunningTotal
FROM Orders;


###  3️⃣ Advanced Subqueries & CTEs

**Concept**

Use subqueries that reference outer queries (correlated), and use Common Table Expressions (CTEs) for readability. Recursive CTEs handle hierarchical data.

In [0]:
-- Correlated subquery: salary above department average
SELECT e.FirstName, e.Salary
FROM Employees e
WHERE Salary > (
  SELECT AVG(Salary)
  FROM Employees e2
  WHERE e2.DepartmentID = e.DepartmentID
);

-- Recursive CTE for org hierarchy
WITH OrgChart AS (
  SELECT EmployeeID, ManagerID, FirstName
  FROM Employees WHERE ManagerID IS NULL
  UNION ALL
  SELECT e.EmployeeID, e.ManagerID, e.FirstName
  FROM Employees e
  INNER JOIN OrgChart o ON e.ManagerID = o.EmployeeID
)
SELECT * FROM OrgChart;


###  4️⃣ Complex Data Manipulation

**Concept**

MERGE, UPSERT and multi-table updates help keep data in sync.

In [0]:
-- Merge example (SQL Server/Oracle)
MERGE INTO TargetTable t
USING SourceTable s
ON t.ID = s.ID
WHEN MATCHED THEN
  UPDATE SET t.Value = s.Value
WHEN NOT MATCHED THEN
  INSERT (ID, Value) VALUES (s.ID, s.Value);


###  5️⃣ Indexing & Performance Basics

**Concept**

Indexes speed up reads but cost storage and maintenance. Use execution plans to identify performance bottlenecks.

In [0]:
-- Create index on DepartmentID
CREATE INDEX IX_Employees_DeptID
ON Employees(DepartmentID);

-- Drop index
DROP INDEX IX_Employees_DeptID ON Employees;


### 6️⃣ Transactions & Error Handling (Deeper)

**Concept**

Use savepoints for partial rollbacks and TRY/CATCH for robust error handling.

In [0]:
BEGIN TRANSACTION;
UPDATE Accounts SET Balance = Balance - 500 WHERE AccountID=1;
SAVE TRANSACTION AfterFirstUpdate;
UPDATE Accounts SET Balance = Balance + 500 WHERE AccountID=2;

-- if error happens
ROLLBACK TRANSACTION AfterFirstUpdate; -- partial rollback
COMMIT;


###  7️⃣ Temporary Tables & Table Variables

**Concept**

Temp tables store intermediate results in complex ETL or reporting scenarios.

In [0]:
CREATE TABLE #TempOrders (OrderID INT, Amount DECIMAL(10,2));
INSERT INTO #TempOrders
SELECT OrderID, Amount FROM Orders WHERE Status='Pending';
SELECT * FROM #TempOrders;
DROP TABLE #TempOrders;


### 8️⃣ Advanced Functions & Expressions

**Concept**

You can write your own functions, pivot data, or use APPLY operators to join table-valued functions.

In [0]:
-- Scalar UDF for tax
CREATE FUNCTION dbo.fnTax(@Amount DECIMAL(10,2))
RETURNS DECIMAL(10,2)
AS
BEGIN
  RETURN @Amount * 0.15;
END;

SELECT OrderID, Amount, dbo.fnTax(Amount) AS Tax
FROM Orders;

-- Pivot (SQL Server example)
SELECT DepartmentID, [2023],[2024]
FROM (
  SELECT DepartmentID, YEAR(HireDate) AS HireYear FROM Employees
) AS SourceTable
PIVOT (COUNT(HireYear) FOR HireYear IN ([2023],[2024])) AS pvt;


### 9️⃣ Working with Complex Data Types (JSON/XML)

**Concept**

Modern SQL supports semi-structured data.

In [0]:
-- JSON (SQL Server)
SELECT JSON_VALUE(JsonColumn,'$.Customer.Name') AS CustomerName
FROM Orders;


### 🔟 Security & Permissions (Advanced)

**Concept**

Organize objects into schemas, grant permissions at role level, implement row-level security.

In [0]:
CREATE SCHEMA Sales AUTHORIZATION dbo;
CREATE ROLE AnalystRole;
GRANT SELECT ON Employees TO AnalystRole;
EXEC sp_addrolemember 'AnalystRole','JohnUser';


### 🔟+1 Capstone Practice

- Build a recursive CTE of your company’s hierarchy.

- Use window functions to find top-N earners per department.

- Create an index to improve a slow query.

- Merge new customer orders from a staging table.

- Write a transaction with savepoints and error handling.

- Create a pivoted view summarizing employees by hire year.